## Import modules and preprocessing

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data import load_data, split_and_prepare_data, split_data
from src.preprocessing import full_pipeline

data_path = Path("..") / "data" / "raw" / "housing.csv"
df = load_data(data_path)

# Preprocessing data
X_train_prep, X_test_prep, y_train, y_test = split_and_prepare_data(df)

## 1. Linear Regression

In [32]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression()

model_1 = reg.fit(X_train_prep, y_train)

In [33]:
from sklearn.metrics import root_mean_squared_error

pred = reg.predict(X_test_prep)

rmse = root_mean_squared_error(y_test, pred)

print(f"rmse: {rmse}")

rmse: 69179.61996170598


## 2. Polynomial Regression

In [34]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

pipline = Pipeline([
    ('poly', PolynomialFeatures(3,)),
    ('lin_reg', LinearRegression())
])

model = pipline.fit(X_train_prep,y_train)

p_pred = model.predict(X_test_prep)

rmse = root_mean_squared_error(y_test, p_pred)

print(f"rmse: {rmse}")



rmse: 411741.19209864864


In [35]:
p_pred = model.predict(X_train_prep)
rmse = root_mean_squared_error(y_train, p_pred)

print(f"rmse: {rmse}")

rmse: 52263.501193787524


## cross validation


In [36]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]
X_train, X_test, y_train, y_test = split_data(X, y)

def px_pipeline(p):
    model = Pipeline([
    ('preprocessing', full_pipeline),
    ('poly', PolynomialFeatures(degree=p, include_bias=False)),
    ('lin_reg', LinearRegression())
    ])
    return model

linear_pipeline = Pipeline([
    ('preprocessing', full_pipeline),
    ('linear', LinearRegression())
])


models = {
    "Linear (deg 1)": linear_pipeline,
    "Poly (deg 2)": px_pipeline(2),
    "Poly (deg 3)": px_pipeline(3),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    out = cross_val_score(
        model,X_train, y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
    )
    
    print(name, -out)
    print("Mean RMSE:", -out.mean())


Linear (deg 1) [66951.39288459 68153.94444058 68201.14644117 66710.71544113
 72147.06151472]
Mean RMSE: 68432.85214443688
Poly (deg 2) [ 60128.99717001 822818.44097025 165918.86479665  63266.23183632
 441156.48446645]
Mean RMSE: 310657.80384793674
Poly (deg 3) [   57050.32755209 18949648.34897381  1036163.22602274   141176.55974658
  3382431.23011673]
Mean RMSE: 4713293.93848239


## Descision forest

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

tree_reg = DecisionTreeRegressor()

tree_model = tree_reg.fit(X_train_prep, y_train)

y_test_tree_pred = tree_model.predict(X_test_prep)
y_train_tree_pred = tree_model.predict(X_train_prep)

test_tree_rmse = root_mean_squared_error(y_test, y_test_tree_pred)
train_tree_rmse = root_mean_squared_error(y_train, y_train_tree_pred)

print(f"train tree rmse: {train_tree_rmse}")
print(f"test tree rmse: {test_tree_rmse}")

train tree rmse: 0.0
test tree rmse: 74389.34873150478


### Tree cross validation

In [42]:
from sklearn.model_selection import cross_val_score, KFold

tree_pipeline = Pipeline([
    ('preprocessing', full_pipeline),
    ('tree', DecisionTreeRegressor())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

out = cross_val_score(
    tree_pipeline,X_train,y_train,
    cv=cv,
    scoring='neg_root_mean_squared_error'
)

print(-out)

[68502.54870065 70094.04935293 71342.96828676 71526.48876517
 71738.57332289]


## Random Forest

In [44]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = RandomForestRegressor()

forest_model = forest_reg.fit(X_train_prep, y_train)

y_test_forest_pred = forest_model.predict(X_test_prep)
y_train_forest_pred = forest_model.predict(X_train_prep)

test_forest_rmse = root_mean_squared_error(y_test, y_test_forest_pred)
train_forest_rmse = root_mean_squared_error(y_train, y_train_forest_pred)

print(f"train forest rmse: {train_forest_rmse}")
print(f"test forest rmse: {test_forest_rmse}")


train forest rmse: 18515.984747211227
test forest rmse: 50691.93469915321


### RandomForest cross validation

In [47]:
from sklearn.model_selection import KFold, cross_val_score

forest_pipeline = Pipeline([
    ('preprocessing', full_pipeline),
    ('forest', RandomForestRegressor())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

forest_scores = cross_val_score(
    forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='neg_root_mean_squared_error'
)

print(f'mean of rmse {forest_scores.mean()}')

mean of rmse -49812.2161106261
